# 01 · RUL benchmark: screening, causal features, ablations

Answers reviewer comments R1-2 (look-ahead), R1-3 (same-cell control), R1-4
(impedance), R1-6 (CIs, 5 seeds) and R2-1 (correlation matrix). Everything runs
from `data/metadata.csv`.

In [1]:
%run ./00_common.ipynb

root: <repository root>


## 1. Screening of all 34 cells

In [2]:
cells = load_cells()
scr = screen(cells)
eligible = scr[scr["eligible"]]["battery"].tolist()
print("cohort (80% SOH EOL):", eligible)
print("uncensored at 70% SOH:", scr[scr["eol70"].notna()]["battery"].tolist())
scr[["battery", "n_cycles", "temp", "q_ref", "eol80", "status"]]

cohort (80% SOH EOL): ['B0005', 'B0006', 'B0007', 'B0018', 'B0036', 'B0042', 'B0043', 'B0044', 'B0054', 'B0055']
uncensored at 70% SOH: ['B0005', 'B0006']


,battery,n_cycles,temp,q_ref,eol80,status
0,B0005,168,24.0,1.856,103.0,eligible
1,B0006,168,24.0,2.035,61.0,eligible
2,B0007,168,24.0,1.891,124.0,eligible
3,B0018,132,24.0,1.855,75.0,eligible
4,B0025,28,24.0,1.849,NaN,excluded: <40 cycles
5,B0026,28,24.0,1.817,NaN,excluded: <40 cycles
6,B0027,28,24.0,1.823,NaN,excluded: <40 cycles
7,B0028,28,24.0,1.805,NaN,excluded: <40 cycles
8,B0029,40,43.0,1.845,NaN,censored (never <80% SOH)
9,B0030,40,43.0,1.782,NaN,censored (never <80% SOH)


In [3]:
pd.DataFrame(HYPERPARAMS, columns=["model", "hyperparameters"]).to_csv(RESULTS / "hyperparameters.csv", index=False)
df_c, cols = feature_table(cells, eligible, "centered")   # submitted (look-ahead) features
df_t, _ = feature_table(cells, eligible, "trailing")      # causal features
df_t.to_csv(RESULTS / "features_trailing.csv", index=False)
print(len(cols), "features,", len(df_t), "labelled cycles")

19 features, 735 labelled cycles


## 2. Impedance missingness per cell (R1-4)

In [4]:
meta = pd.read_csv(ROOT / "data" / "metadata.csv")
n_imp = meta[meta["type"] == "impedance"].groupby("battery_id").size()
miss = pd.DataFrame([dict(battery=b, temp=cells[b]["temp"], q_ref=round(cells[b]["qref"], 3),
                          eol80=cells[b]["eol80"], n_impedance_tests=int(n_imp.get(b, 0)),
                          Re_missing_frac=df_t[df_t.battery_id == b]["Re"].isna().mean(),
                          Rct_missing_frac=df_t[df_t.battery_id == b]["Rct"].isna().mean())
                     for b in eligible])
miss.to_csv(RESULTS / "impedance_missingness.csv", index=False)
miss.round(3)

,battery,temp,q_ref,eol80,n_impedance_tests,Re_missing_frac,Rct_missing_frac
0,B0005,24.0,1.856,103,278,0.097,0.097
1,B0006,24.0,2.035,61,278,0.176,0.176
2,B0007,24.0,1.891,124,278,0.079,0.079
3,B0018,24.0,1.855,75,53,0.000,0.000
4,B0036,24.0,1.985,189,92,0.000,0.000
5,B0042,22.0,1.731,62,50,0.000,0.000
6,B0043,22.0,1.714,57,50,0.000,0.000
7,B0044,22.0,1.687,54,50,0.000,0.000
8,B0054,4.0,1.167,55,48,0.000,0.000
9,B0055,4.0,1.320,55,48,0.000,0.000


## 3. Feature correlation matrix (R2-1)

In [5]:
corr = df_t[cols + ["rul"]].corr(method="spearman")
corr.to_csv(RESULTS / "feature_corr_spearman.csv")
lags = ["soh"] + [f"soh_lag{i}" for i in range(1, 11)]
print("min rho among SOH lags:", corr.loc[lags, lags].values.min().round(3))
print("T_amb vs Q_ref:", round(corr.loc["temp", "q_ref"], 3), "| Re vs Rct:", round(corr.loc["Re", "Rct"], 3))

min rho among SOH lags: 0.888
T_amb vs Q_ref: 0.818 | Re vs Rct: 0.803


## 4. LOBO: centered vs causal features, with and without impedance, 5 seeds
Fixed (`base`) configuration. The nested selection that produces Table II is in notebook 02.

In [6]:
def lobo_base(df, cc, seed):
    preds = {}
    for held in eligible:
        train, test = df[df.battery_id != held], df[df.battery_id == held].sort_values("cycle")
        Xtr, ytr, med = xy(train, cc)
        Xte, yte, _ = xy(test, cc, med)
        for name, model in get_models(seed).items():
            model.fit(Xtr, ytr)
            preds.setdefault(name, {})[held] = dict(cycle=test["cycle"].to_numpy(),
                                                    rul_pred=np.clip(model.predict(Xte), 0, None), rul_true=yte)
    return preds

t0 = time.time()
parts = []
for tag, df, cc in [("centered_submitted", df_c, cols), ("trailing", df_t, cols),
                    ("trailing_noImpedance", df_t, [c for c in cols if c not in IMPEDANCE])]:
    for seed in SEEDS:
        e = per_cell_errors(lobo_base(df, cc, seed), cells); e["config"] = tag; e["seed"] = seed
        parts.append(e)
    print(f"{tag} done ({time.time() - t0:.0f}s)")
errs = pd.concat(parts, ignore_index=True)
errs.to_csv(RESULTS / "per_cell_errors.csv", index=False)

centered_submitted done (62s)


trailing done (111s)


trailing_noImpedance done (160s)


In [7]:
rows = []
for tag in errs["config"].unique():
    s = summarise(errs[(errs.config == tag) & (errs.seed == SEED)], tag)
    sd = (errs[errs.config == tag].groupby(["method", "lambda", "seed"])["abs_err"].mean()
          .groupby(["method", "lambda"]).agg(["mean", "std"]).reset_index()
          .rename(columns={"mean": "MAE_5seed_mean", "std": "MAE_5seed_sd"}))
    rows.append(s.merge(sd, on=["method", "lambda"]))
summ = pd.concat(rows, ignore_index=True)
summ.to_csv(RESULTS / "lobo_summary.csv", index=False)
summ[summ["lambda"] == 0.7][["config", "method", "MAE", "MAE_ci_lo", "MAE_ci_hi", "MAE_5seed_sd", "HR", "alpha_acc"]].round(2)

,config,method,MAE,MAE_ci_lo,MAE_ci_hi,MAE_5seed_sd,HR,alpha_acc
1,centered_submitted,LightGBM,14.36,7.97,22.35,0.29,0.4,0.2
4,centered_submitted,Linear,25.44,15.81,36.44,0.00,0.1,0.0
7,centered_submitted,RandomForest,12.82,7.08,19.85,0.09,0.4,0.2
10,centered_submitted,Ridge,21.68,13.23,31.17,0.00,0.3,0.0
13,centered_submitted,XGBoost,13.20,8.02,19.87,0.25,0.4,0.2
16,trailing,LightGBM,13.06,6.60,21.24,0.33,0.6,0.2
19,trailing,Linear,21.23,11.59,32.98,0.00,0.3,0.1
22,trailing,RandomForest,12.25,5.84,19.89,0.24,0.6,0.4
25,trailing,Ridge,12.36,7.76,17.92,0.00,0.5,0.2
28,trailing,XGBoost,11.82,5.87,18.74,0.78,0.6,0.3


In [8]:
w = wilcoxon_table(errs[(errs.config == "trailing") & (errs.seed == SEED)])
w.to_csv(RESULTS / "wilcoxon_trailing_l07.csv", index=False)
w.round(3)

,a,b,mean_a,mean_b,p
0,LightGBM,Linear,13.057,21.234,0.027
1,LightGBM,RandomForest,13.057,12.248,0.375
2,LightGBM,Ridge,13.057,12.361,0.695
3,LightGBM,XGBoost,13.057,11.816,0.193
4,Linear,RandomForest,21.234,12.248,0.020
5,Linear,Ridge,21.234,12.361,0.109
6,Linear,XGBoost,21.234,11.816,0.014
7,RandomForest,Ridge,12.248,12.361,0.922
8,RandomForest,XGBoost,12.248,11.816,0.846
9,Ridge,XGBoost,12.361,11.816,0.922
